# K-means binarization and IceDetectionAlgorithms
Here, we look at the details of the $k$-means binarization method and the use of `IceDetectionAlgorithms`.

In [ ]:
using Pkg
Pkg.add("IceFloeTracker")
Pkg.add("Images")

First we load necessary packages. We'll use examples from the `DataLoader`.

In [ ]:
using IceFloeTracker
using Images

## 1. Load data
For this example, we'll use a subset of a large image from the western Arctic Ocean. We need the true color, false color, and land mask images. We can use `mosaicview` to view the images side-by-side. The true color image uses MODIS bands 1, 4, and 3 (red, green, blue) and approximates the scene perceived by a human eye. The false color image uses MODIS bands 7, 2, and 1 (near infrared, shortwave infrared, and red).

In [ ]:
dataset = Watkins2026Dataset(; ref="v0.1")
cases = filter(c -> c.case_number ∈ [6, 63, 112] && c.satellite == "aqua", dataset)
cases

In [ ]:
truecolor_imgs = modis_truecolor.(cases)
mosaicview(truecolor_imgs, nrow=1)

## 2. Preprocessing
These scenes each have prominent floes amidst mixed sea ice and water, and differ in the brightness of the prominent floes. Before applying the $k$-means segmentation, we recommend preprocessing the image to enhance contrast and remove land and cloud. These images don't contain land, so we'll focus on the clouds and contrast adjustment.

In [ ]:
falsecolor_imgs = modis_falsecolor.(cases)
cloudmasks = Watkins2025CloudMask().(falsecolor_imgs)
mosaicview(vcat(falsecolor_imgs, [Gray.(cm) for cm in cloudmasks]), nrow=2, rowmajor=true)

In [ ]:
import IceFloeTracker: ContrastLimitedAdaptiveHistogramEqualization as CLAHE
function preprocess(tc_img, cldmsk)
    proc_img = adjust_histogram(Gray.(tc_img), CLAHE(nbins=256, rblocks=2, cblocks=2, clip=5))
    proc_img .= unsharp_mask(proc_img, 100, 0.2, 0.01)
    proc_img .= nonlinear_diffusion(proc_img, PeronaMalikDiffusion())
    apply_cloudmask!(proc_img, cldmsk)
    return proc_img
end
preproc_imgs = preprocess.(truecolor_imgs, cloudmasks)
mosaicview(preproc_imgs, nrow=1)

# $k$-means segmentation
The $k$-means binarization function uses the clustering method internally. Here we'll use the `kmeans_segmentation` function directly and use `view_seg_random` to visualize the output, then show how `IceDetectionAlgorithms` aid cluster selection for binarization.

In [ ]:
kmeans_seg = kmeans_segmentation.(preproc_imgs; k=4)

In [ ]:
mosaicview(view_seg_random.(kmeans_seg), nrow=1)

Some things to note: The $k$-means algorithm assigns the labels randomly, so cluster 1 in one image does not generally correspond with cluster 1 in a different image. Here we can see some common characteristics:
1. In the first image, the bright floes are nearly all in the same cluster. A secondary cluster primarily makes up the borders of floes. 
2. In the second image, the floes are more heterogeneous - there are melt ponds on the surface, and differing colors of grays. Most of the background is one color, but this image likely will not be segmented will with k-means.
3. This image does separate neatly, with all the prominent floes being bright white. However there also are portions of the ice filiments that are in the same cluster, so additional work to separate out the bright floes is needed.

# 3. IceDetectionAlgorithms
`IceDetectionAlgorithms` in IFT aim to identify pixels with the brightest ice. They are compatible with Julia Images `binarize` command. In general they leverage the false color images, which have near and shortwave infrared bands. These bands are lower resolution than the visible blue and green bands, however can show stronger differences between sea ice and water. We'll show two versions:
1. Simple thresholds via `IceDetectionThresholdMODIS721`
2. Histogram-based threshold via `IceDetectionBrightnessPeaksMODIS721`

In [ ]:
falsecolor_imgs = modis_falsecolor.(cases)
falsecolor_masked = [apply_cloudmask(RGB.(fc), cm) for (fc, cm) in zip(falsecolor_imgs, cloudmasks)]

simple_threshold_algorithm = IceDetectionThresholdMODIS721(
                band_7_max=0.1,
                band_1_min=0.7,
                band_2_min=0.7
                )
adaptive_threshold_algortihm = IceDetectionBrightnessPeaksMODIS721(
                band_7_max=0.1, 
                possible_ice_threshold=0.3,
                join_method="union"
                )

simple_threshold_binarized = [binarize(fm, simple_threshold_algorithm) for fm in falsecolor_masked]
adaptive_threshold_binarized = [binarize(fm, adaptive_threshold_algortihm) for fm in falsecolor_masked]

mosaicview(vcat(simple_threshold_binarized, adaptive_threshold_binarized), nrow=2, rowmajor=true)

# $k$-means Binarization

Finally, we put together the k-means segmentation and the IceDetectionAlgorithms to produce a binarized image. The method works by identifying the cluster in the k-means segmentation result which contains the highest fraction of the flagged pixels in the IceDetectionAlgorithm output.

In [ ]:
kmeans_binarized_simple = kmeans_binarization.(preproc_imgs, falsecolor_masked;
    k=4, cluster_selection_algorithm=simple_threshold_algorithm)
kmeans_binarized_adaptive = kmeans_binarization.(preproc_imgs, falsecolor_masked;
    k=4, cluster_selection_algorithm=adaptive_threshold_algortihm);

Comparison of the kmeans segmentation and the simple binarization results:

In [ ]:
mosaicview(vcat(view_seg_random.(kmeans_seg),
 [Gray.(b) for b in kmeans_binarized_simple],
  [Gray.(b) for b in kmeans_binarized_adaptive]), nrow=3, rowmajor=true)

# Next steps
Both methods for cluster detection performed well in this case. The results are identical because the same $k$-means cluster was selected in each case, even though the intermediate images from the IceDetectionAlgorithms differ. In this case, parameters were selected for the detection algorithms to give both a fair shot; in practice, the adaptive algorithm may be a better choice if the statistics of ice floe reflectance are not known. 

Following the creation of the binarized image, the user will need to perform further processing to fill holes, clean up edges, separate ice floes, and filter out filaments and other non-floe objects.